# 01 Crosswalk

**Owner:** Together, pair programming (Yair types, Nicolas checks)

**What this notebook does:** builds two small lookup tables that translate UCDP country codes and SIPRI country names into the code `iso3` we use everywhere.

**Reads:** `data/clean/vdem.csv` (from 03), UCDP and SIPRI raw files

**Writes:** `crosswalk_gw.csv`, `crosswalk_sipri.csv`

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Write simple code in small steps: one operation per line, print the result after each step. This one has no sandbox: you write it together directly here. Run it after 03 V-Dem is done.

## Setup

In [ ]:
import os
import pandas as pd
import plotly.express as px

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

## Why we need this

The three datasets name countries differently. V-Dem uses `country_text_id` (for example `UKR`) and a COW number.
UCDP uses Gleditsch-Ward numbers (for example 369). SIPRI uses English names (for example "Ukraine").
Before we can merge anything, every source needs the same key: `iso3`.

Note: `iso3` is V-Dem's own code. For most countries it is the official ISO code, but a few are V-Dem codes
(for example XKX Kosovo, PSG Gaza, SML Somaliland, DDR East Germany, YMD South Yemen, VDR South Vietnam).
That is fine for merging. Plotly maps cannot draw those few, which is why some countries will be missing on the world map later.

---
## Part 1: The list of countries from V-Dem

**Step 1.** Load the clean V-Dem file `CLEAN + "vdem.csv"` with only three columns: `usecols=["country_name", "iso3", "cow_code"]`. Call it `vdem_names`. Print the shape. (We use the clean file, not the raw one, so only countries and years from our panel are included.)

In [ ]:
# your code here

**Step 2.** V-Dem has one row per country per year, so every country appears many times. Keep one row per combination of `iso3` and `cow_code` with `drop_duplicates(subset=["iso3", "cow_code"])`. Print the shape again. Roughly how many countries are there?

In [ ]:
# your code here

**Step 3.** Print `vdem_names[vdem_names["iso3"].duplicated()]`. If an iso3 appears twice, its COW code changed over time. Write down both codes in a comment.

In [ ]:
# your code here

**Step 4.** Print the rows for Germany, Yemen, Vietnam and Serbia (`vdem_names["country_name"].str.contains("Germany")` and so on). Note each `cow_code`. You compare them with the GW list in step 8.

In [ ]:
# your code here

---
## Part 2: crosswalk_gw (UCDP codes → iso3)

**Step 5.** Start simple: for most countries the Gleditsch-Ward number is the same as the COW number. Make a new table `crosswalk_gw` with two columns: `gw_code` (a copy of `cow_code`) and `iso3`. Drop rows where `cow_code` is missing. Make `gw_code` an integer. If one `gw_code` appears twice now, decide which iso3 is right and drop the other row.

In [ ]:
# your code here

In [ ]:
# your code here

**Step 6.** Collect every code that appears in UCDP from 1950 on. Load UCDP into `ucdp_raw` and keep only rows with `year >= 1950` (our panel starts in 1950; earlier codes can belong to states V-Dem does not have). Then loop over the column `gwno_loc`: turn each value into text, split it on the comma, strip spaces, turn each piece into `int`, and add it to a list `ucdp_codes`. At the end, keep each code once (`set(ucdp_codes)`). Print how many different codes there are.

In [ ]:
# your code here

In [ ]:
# your code here

**Step 7.** Find the codes that are not yet in `crosswalk_gw`: loop over your set, and if a code is not in `list(crosswalk_gw["gw_code"])`, add it to a list `missing_gw`. Print it.

In [ ]:
# your code here

**Step 8.** Decide which `iso3` each missing code belongs to (UCDP codebook or the Gleditsch-Ward country list online). Write them in a dictionary `manual_gw = {code: "ISO", ...}`. Then loop over the dictionary and add one row per entry to `crosswalk_gw` (build a small dataframe from a list of dictionaries and use `pd.concat`).

In [ ]:
# your code here

In [ ]:
# your code here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

GW and COW differ for a few countries. Check at least these by printing the matching crosswalk rows:
Yemen (GW 678 is North Yemen and, after 1990, unified Yemen; COW uses 679 after 1990),
Germany (GW 260 is West Germany and also Germany after 1990; COW uses 255 after 1990),
South Yemen 680, Vietnam 816 and South Vietnam 817, Yugoslavia/Serbia 345, Kosovo 347, USSR/Russia 365.
Expect only a handful of missing codes.

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

Adding rows the simple way:

```
new_rows = []
for code in manual_gw:
    new_rows.append({"gw_code": code, "iso3": manual_gw[code]})
crosswalk_gw = pd.concat([crosswalk_gw, pd.DataFrame(new_rows)])
```
If a code is already in the table with the wrong iso3, remove that row first by filtering it out.

</details>

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
assert crosswalk_gw["gw_code"].dtype.kind == "i"
assert crosswalk_gw["gw_code"].duplicated().sum() == 0, "each gw_code only once"
assert crosswalk_gw["iso3"].isna().sum() == 0
still_missing = [code for code in set(ucdp_codes) if code not in list(crosswalk_gw["gw_code"])]
print("UCDP codes still without iso3:", still_missing)
assert len(still_missing) == 0
for code in [2, 255, 260, 365, 666, 678, 750]:
    print(code, list(crosswalk_gw[crosswalk_gw["gw_code"] == code]["iso3"]))
print("OK (now check the printed iso3 codes by eye)")

---
## Part 3: crosswalk_sipri (SIPRI names → iso3)

**Step 9.** Load SIPRI (`skiprows=11`, `encoding="latin-1"`). Make a list of the different recipient names: `all_names = list(sipri_raw["Recipient"].unique())`. Loop over it and keep, in a new list called `sipri_names`, only names that do not end with `*` and are not "unknown recipient(s)" (print the names containing "unknown" first to see the exact spelling). Strip spaces from each name with `name.strip()`. Print how many are left.

In [ ]:
# your code here

In [ ]:
# your code here

**Step 10.** Compare with V-Dem names. Loop over `sipri_names`: if the name is in `list(vdem_names["country_name"])`, add it to `matched`, otherwise add it to `unmatched`. Print both lengths and the full `unmatched` list. Expect roughly 25 to 40 unmatched names.

In [ ]:
# your code here

**Step 11.** For every unmatched name, find the right `iso3` by hand (search V-Dem for a similar name, for example `vdem_names[vdem_names["country_name"].str.contains("Congo")]`). Historical states map to the V-Dem unit of those years: Soviet Union to RUS, Yugoslavia to SRB, East Germany to DDR, North Yemen to YEM, South Yemen to YMD, South Vietnam to VDR (check each in `vdem_names`). Write them in a dictionary `manual_sipri = {"SIPRI name": "ISO", ...}`. If a name really has no V-Dem unit, leave it out and write it in the decisions table; 04 drops those rows on purpose.

In [ ]:
# your code here

In [ ]:
# your code here

**Step 12.** Build the table: an empty list `rows`. For every name in `matched`, look up its `iso3` in V-Dem and append `{"recipient": name, "iso3": code}`. For every name in `manual_sipri`, append the same. Then `crosswalk_sipri = pd.DataFrame(rows)`.

In [ ]:
# your code here

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

To look up one name: `vdem_names[vdem_names["country_name"] == name]["iso3"].iloc[0]`. `.iloc[0]` takes the first value. A name can appear twice in `vdem_names` (two COW codes, step 3); both rows have the same iso3, so `.iloc[0]` is fine.

</details>

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
assert list(crosswalk_sipri.columns) == ["recipient", "iso3"]
assert crosswalk_sipri["recipient"].duplicated().sum() == 0, "each name only once"
assert crosswalk_sipri["iso3"].isna().sum() == 0
for name in ["Soviet Union", "India", "Ukraine"]:
    print(name, list(crosswalk_sipri[crosswalk_sipri["recipient"] == name]["iso3"]))
print(len(crosswalk_sipri), "SIPRI names matched")
print("OK (check the printed codes by eye)")

## Save

**Step 13.** Save both tables to `data/clean/` with `index=False`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
assert os.path.exists(CLEAN + "crosswalk_gw.csv")
assert os.path.exists(CLEAN + "crosswalk_sipri.csv")
print("OK")

## Decisions in this notebook

Write every choice you made here (and copy it into the decisions log in `CONVENTIONS.md`). You need these for the report.

| Decision | Why |
| --- | --- |
|  |  |